In [2]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path("../")
METADATA_PATH = BASE_DIR / "metadata" / "metadata.jsonl"

ACTUAL_FIELDS = ["atap", "dinding", "lantai"]

print("Metadata path     :", METADATA_PATH.resolve())

Metadata path     : C:\Users\Lutfi\Documents\Project\AITF\rutilahu-vlm-etl\metadata\metadata.jsonl


In [3]:
def load_metadata_jsonl(path: Path) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    return df


def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        v = value.strip()
        return v if v else None
    return str(value).strip()


def tidy_metadata(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "actual_label" in out.columns:
        actual = pd.json_normalize(out["actual_label"])
        actual.columns = [f"actual_{c}" for c in actual.columns]
        out = pd.concat([out.drop(columns=["actual_label"]), actual], axis=1)

    rename_map = {
        "actual_atap": "atap",
        "actual_dinding": "dinding",
        "actual_lantai": "lantai",
    }
    out = out.rename(columns=rename_map)

    keep_cols = ["house_id"] + [c for c in ACTUAL_FIELDS if c in out.columns]
    out = out[keep_cols].copy()
    for c in ACTUAL_FIELDS:
        if c in out.columns:
            out[c] = out[c].map(normalize_label)
    return out

In [4]:
# Load data
df_md_raw = load_metadata_jsonl(METADATA_PATH)

print("Metadata rows     :", len(df_md_raw))

df_md = tidy_metadata(df_md_raw)

display(df_md.head())

Metadata rows     : 59182


,house_id,atap,dinding,lantai
0,H00001,genteng,tembok,semen/bata_merah
1,H00002,genteng,tembok,ubin/tegel/teraso
2,H00003,genteng,tembok,semen/bata_merah
3,H00004,genteng,tembok,keramik
4,H00005,genteng,tembok,keramik


In [6]:
# Unique labels from actual_label (metadata)
for col in ACTUAL_FIELDS:
    if col in df_md.columns:
        print(f"\n=== Unique label: {col} ===")
        uniq = sorted(df_md[col].dropna().unique().tolist())
        print(uniq)
        print("Count unique:", len(uniq))


=== Unique label: atap ===
['asbes', 'bambu', 'beton', 'genteng', 'jerami/ijuk/daun-daunan/rumbia', 'kayu/sirap', 'lainnya', 'seng', 'unclassified']
Count unique: 9

=== Unique label: dinding ===
['anyaman_bambu', 'bambu', 'batang_kayu', 'kayu/papan/gypsum/GRC/calciboard', 'lainnya', 'plesteran_anyaman_bambu/kawat', 'tembok', 'unclassified']
Count unique: 8

=== Unique label: lantai ===
['bambu', 'kayu/papan', 'keramik', 'lainnya', 'marmer/granit', 'parket/vinil/karpet', 'semen/bata_merah', 'tanah', 'ubin/tegel/teraso', 'unclassified']
Count unique: 10


In [18]:
def value_counts_table(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    frames = []
    for col in cols:
        if col in df.columns:
            vc = df[col].value_counts(dropna=False).reset_index()
            vc.columns = [col, "count"]
            vc.insert(0, "field", col)
            frames.append(vc)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

print("=== Distribusi label Label Studio ===")
display(value_counts_table(df_ls, LABEL_FIELDS))

print("=== Distribusi label actual_label ===")
display(value_counts_table(df_md, ACTUAL_FIELDS))

=== Distribusi label Label Studio ===


,field,jenis_atap_terluas,count,jenis_dinding_terluas,jenis_lantai_terluas
0,jenis_atap_terluas,Tidak terdeteksi,2783,NaN,NaN
1,jenis_atap_terluas,Genteng,2242,NaN,NaN
2,jenis_atap_terluas,Asbes,783,NaN,NaN
3,jenis_atap_terluas,Seng,354,NaN,NaN
4,jenis_atap_terluas,Beton,133,NaN,NaN
5,jenis_atap_terluas,Jerami/ijuk/daun-daunan/rumbia,13,NaN,NaN
6,jenis_atap_terluas,Kayu/sirap,8,NaN,NaN
7,jenis_atap_terluas,Lainnya,3,NaN,NaN
8,jenis_atap_terluas,Bambu,2,NaN,NaN
9,jenis_dinding_terluas,NaN,4499,Tembok,NaN


=== Distribusi label actual_label ===


,field,atap,count,dinding,lantai
0,atap,Genteng,13079,NaN,NaN
1,atap,Tidak terdeteksi,4058,NaN,NaN
2,atap,Asbes,1729,NaN,NaN
3,atap,Seng,599,NaN,NaN
4,atap,Beton,421,NaN,NaN
5,atap,unclassified,93,NaN,NaN
6,atap,Lainnya,83,NaN,NaN
7,atap,Bambu,30,NaN,NaN
8,atap,Jerami/ijuk/daun-daunan/rumbia,14,NaN,NaN
9,atap,Kayu/sirap,12,NaN,NaN
